In [ ]:
# | default_exp features.fsc_binlevel_genomewide

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()


In [ ]:
# | export
class FSCGenomewideEvaluator(FeatureEvaluator):
    """Extracts GC-corrected GC log2 signals from genomic bins."""

    name = "FSCGenomewide"
    source_file = ".FSC.parquet"
    tier = 1
    category = "fragmentation"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted

            cols = set(df.columns)
            log2_cols = [c for c in cols if c.endswith("_log2")]

            # Global median over all bins
            for l in log2_cols:
                extracted[f"global_{l}_median"] = float(df[l].median())

            # Per-chromosome aggregates
            if "chrom" in cols:
                chroms = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]
                for c in chroms:
                    sub = df[df["chrom"] == c]
                    if sub.empty:
                        continue
                    for l in log2_cols:
                        extracted[f"{c}_{l}_median"] = float(sub[l].median())

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
evaluator = FSCGenomewideEvaluator()
df_test = pd.DataFrame(
    [{"chrom": "chr1", "core_short_log2": -2.1, "ultra_short_log2": 1.1}]
)
res = evaluator.extract(df_test)
assert "global_core_short_log2_median" in res
assert "chr1_ultra_short_log2_median" in res